### Set up your Google API Key

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`.

### Set up your Google API Key

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`.

### Set up your Google API Key

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`.

In [37]:
import sys
!{sys.executable} -m pip install -U langchain-google-genai

# Import necessary packages
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Get the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

In [38]:
# Example interaction
response = llm.invoke("Write a short, catchy slogan for a real estate company that specializes in unique premium homes.")
print(response.content)


Here are a few options, playing on different angles of "unique" and "premium":

**Short & Punchy:**

1.  **Your Extraordinary Address.**
2.  **Uniquely Premium. Perfectly Yours.**
3.  **Curating the Exceptional.**
4.  **Beyond Ordinary. Beyond Compare.**
5.  **The Art of Unique Luxury.**

**Slightly More Descriptive:**

6.  **Distinctive Homes. Elevated Living.**
7.  **Luxury Redefined. Uniquely Yours.**
8.  **Where Luxury Finds Its Match.**

Choose the one that best reflects the specific brand personality!


## Langchain Ingestion and Retrieval Workflow

We will now set up a Retrieval-Augmented Generation (RAG) workflow. This involves ingesting a document into a vector store and then using an LLM to answer questions based on the retrieved information.

In [39]:
# Install additional necessary packages
%pip install -U chromadb tiktoken

### Create a Sample Transport Policy Document

I will create a dummy text for demonstration purposes. In a real scenario, you would load your actual document from a file (e.g., PDF, TXT, DOCX).

In [40]:
# Define a dummy Transport Policy Document
transport_policy_text = """# Transport Policy Handbook

## Section 1: Company Vehicle Usage

Company vehicles are provided for official business use only. Employees authorized to drive company vehicles must possess a valid driver's license and adhere to all traffic laws. Personal use of company vehicles is strictly prohibited. Any damage or malfunction of a company vehicle must be reported immediately to the Fleet Manager.

## Section 2: Business Travel Reimbursement

### 2.1 Approved Modes of Transport

Employees traveling for business are encouraged to use the most cost-effective and efficient modes of transport, including public transportation, carpooling, or economy class air travel. Business class travel requires prior approval from a Senior Director.

### 2.2 Mileage Reimbursement

For use of personal vehicles for business travel, employees will be reimbursed at the standard company mileage rate. Mileage claims must be submitted with a clear record of distance traveled and purpose of journey. Commuting to and from the primary place of work is not eligible for reimbursement.

## Section 3: Commuting Support

### 3.1 Public Transport Pass Scheme

The company offers a subsidized public transport pass scheme for employees. Details and application forms are available from HR. This scheme aims to reduce carbon footprint and support sustainable commuting options.

### 3.2 Parking Facilities

Limited parking facilities are available on-site for employees. Priority is given to carpoolers and employees with special needs. All vehicles parked on company property must display a valid parking permit.

## Section 4: Safety Guidelines

All employees engaged in business travel or using company transport must prioritize safety. This includes taking adequate rest breaks, avoiding driving under the influence, and ensuring vehicles are roadworthy. Incident reporting procedures are detailed in Appendix A."""

# Create a Document object directly from the string
transport_documents = [Document(page_content=transport_policy_text, metadata={"source": "Transport Policy Handbook"})]

print(f"Loaded {len(transport_documents)} transport document.")
print("First 200 characters of the transport document:")
print(transport_documents[0].page_content[:200])

Loaded 1 transport document.
First 200 characters of the transport document:
# Transport Policy Handbook

## Section 1: Company Vehicle Usage

Company vehicles are provided for official business use only. Employees authorized to drive company vehicles must possess a valid driv


### Split the Document into Chunks

Large documents need to be split into smaller, manageable chunks so that the embeddings are more granular and relevant information can be retrieved effectively.

In [41]:
%pip install -U langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Max size of each chunk
    chunk_overlap=200, # Overlap between chunks to maintain context
    length_function=len,
    add_start_index=True,
)

# Combine only the Transport document and split it into chunks
all_documents = transport_documents
chunks = text_splitter.split_documents(all_documents)

print(f"Split documents into {len(chunks)} chunks.")
print("First chunk content:")
print(chunks[0].page_content)

Split documents into 3 chunks.
First chunk content:
# Transport Policy Handbook

## Section 1: Company Vehicle Usage

Company vehicles are provided for official business use only. Employees authorized to drive company vehicles must possess a valid driver's license and adhere to all traffic laws. Personal use of company vehicles is strictly prohibited. Any damage or malfunction of a company vehicle must be reported immediately to the Fleet Manager.

## Section 2: Business Travel Reimbursement

### 2.1 Approved Modes of Transport

Employees traveling for business are encouraged to use the most cost-effective and efficient modes of transport, including public transportation, carpooling, or economy class air travel. Business class travel requires prior approval from a Senior Director.

### 2.2 Mileage Reimbursement


### Initialize Google Generative AI Embeddings

We need an embedding model to convert our text chunks into numerical vectors. These vectors are then stored in the vector database and used for similarity search.

In [42]:
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the embedding model
# Make sure GOOGLE_API_KEY is available from previous cells
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GOOGLE_API_KEY)

print("Google Generative AI Embeddings initialized.")

Google Generative AI Embeddings initialized.


### Create and Populate Chroma Vector Store

Now, we'll use Chroma to store our document chunks and their corresponding embeddings. Chroma will enable us to quickly find relevant document parts when a query is made.

In [43]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Get the API key (ensure it's available for this cell's execution)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Re-initialize the embedding model with 'models/embedding-001'
# Both 'models/embedding-001' and 'text-embedding-004' are failing with NOT_FOUND errors.
# Reverting to 'models/embedding-001' as it was the model that initialized successfully in a prior cell.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2", google_api_key=GOOGLE_API_KEY)

# Create a Chroma vector store from the document chunks and embeddings
# We'll create an in-memory Chroma instance for this demonstration.
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Chroma vector store created with {vectorstore._collection.count()} entries.")

# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

print("Retriever created from Chroma vector store.")

Chroma vector store created with 11 entries.
Retriever created from Chroma vector store.


### Set up the Retrieval-Augmented Generation (RAG) Chain

Finally, we'll combine our retriever with the `gemini-2.5-flash` LLM to create a RAG chain. This chain will retrieve relevant document chunks based on a query and then use the LLM to generate an answer informed by those chunks.

In [44]:
# The original %pip install command is removed to avoid dependency conflicts and is not needed for a functional RAG setup.
# The original imports for chain functions are removed as per the request to write code without chains.
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template for our LLM
# The context will be filled by the retrieved documents
prompt = ChatPromptTemplate.from_template(
    """Answer the user's question based on the provided context.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context: {context}
    Question: {input}"""
)

# Define a custom class to simulate the behavior of the retrieval_chain
# without using Langchain's chain objects.
class CustomRetrievalChain:
    def invoke(self, input_dict):
        # Assumes 'llm' and 'retriever' are available in the global scope
        # from previously executed cells.
        question = input_dict["input"]

        # 1. Retrieve relevant documents using the retriever
        retrieved_docs = retriever.invoke(question)

        # 2. Format the retrieved documents into a single context string
        #    Each document's page_content is joined by double newlines.
        formatted_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

        # 3. Format the prompt with the context and the user's question
        #    ChatPromptTemplate.format_messages returns a list of Message objects.
        formatted_prompt_messages = prompt.format_messages(
            context=formatted_context,
            input=question
        )

        # 4. Invoke the LLM with the formatted prompt messages
        response = llm.invoke(formatted_prompt_messages)

        # The result should match the structure of the original retrieval_chain's output.
        return {"answer": response.content}

# Instantiate the custom retrieval chain, so subsequent cells can call retrieval_chain.invoke()
retrieval_chain = CustomRetrievalChain()

print("RAG 'retrieval_chain' (manual implementation) created successfully.")

RAG 'retrieval_chain' (manual implementation) created successfully.


In [45]:
sample_question = "What is the policy for company vehicle usage?"
print(f"\nQuestion: {sample_question}")
response_sample = retrieval_chain.invoke({"input": sample_question})
print(f"\nAnswer: {response_sample['answer']}")


Question: What is the policy for company vehicle usage?

Answer: Company vehicles are provided for official business use only. Employees authorized to drive them must possess a valid driver's license and adhere to all traffic laws. Personal use of company vehicles is strictly prohibited. Any damage or malfunction must be reported immediately to the Fleet Manager.


### Demonstrate Retrieval

Now, let's ask a question about the transport policy and see how the RAG chain responds.

In [46]:
# Ask a question about the transport policy
transport_question = "What is the policy for using personal vehicles for business travel?"
print(f"\nQuestion: {transport_question}")
transport_response = retrieval_chain.invoke({"input": transport_question})
print(f"\nAnswer: {transport_response['answer']}")


Question: What is the policy for using personal vehicles for business travel?

Answer: For use of personal vehicles for business travel, employees will be reimbursed at the standard company mileage rate. Mileage claims must be submitted with a clear record of distance traveled and purpose of journey. Commuting to and from the primary place of work is not eligible for reimbursement.
